<a href="https://colab.research.google.com/github/VincenzoManto/HistWarDB/blob/main/HistWarDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import datetime
import re
import pandas as pd
import requests
import plotly.graph_objects as go
from google.colab import files

class HistWarDB:
    """
    Advanced dynamic dataset based on Wikidata.
    Supports sovereign and historical states, anti-noise filters, LIKE search, and analytical views.
    """

    def __init__(self, start_year=1700, end_year=None):
        self.start_year = start_year
        self.end_year = end_year if end_year else datetime.datetime.now().year
        self.df_long = None
        self.fetch_data()

    def _parse_year(self, date_str):
        if not date_str or not isinstance(date_str, str):
            return None
        match = re.search(r'([+-]?\d+)', date_str)
        return int(match.group(1)) if match else None

    def fetch_data(self):
        url = 'https://query.wikidata.org/sparql'
        headers = {'User-Agent': 'FastWikidataWarDataset/17.3 (colab-research)'}

        chunk_size = 25  # 25-year blocks to handle temporal overlaps
        all_raw = []

        print(f"⏳ Advanced extraction with rigorous validation from Wikidata ({self.start_year}-{self.end_year})...")

        for chunk_start in range(self.start_year, self.end_year, chunk_size):
            chunk_end = min(chunk_start + chunk_size, self.end_year)

            sparql_query = f"""
            SELECT DISTINCT ?conflictLabel ?countryLabel ?startDate ?endDate ?countryStart ?countryEnd ?countryEndFallback WHERE {{
                ?conflict wdt:P31/wdt:279* wd:Q198 ;
                        wdt:P710 ?country ;
                        wdt:P580 ?startDate .

                OPTIONAL {{ ?conflict wdt:P582 ?endDate . }}

                # Exclusion of peacekeeping missions and military exercises
                MINUS {{ ?conflict wdt:P31/wdt:279* wd:Q35349 . }}
                MINUS {{ ?conflict wdt:P31/wdt:279* wd:Q1151430 . }}

                # Inclusion of sovereign states AND historical states
                ?country wdt:P31/wdt:279* ?countryType .
                VALUES ?countryType {{ wd:Q3624078 wd:Q3024240 wd:Q6256 }}

                # State existence dates (including fallbacks)
                OPTIONAL {{ ?country wdt:P571 ?countryStart . }}
                OPTIONAL {{ ?country wdt:P576 ?countryEnd . }}
                OPTIONAL {{ ?country wdt:P582 ?countryEndFallback . }}

                # Exclusion of non-state entities
                MINUS {{ ?country wdt:P31/wdt:279* wd:Q5 . }}
                MINUS {{ ?country wdt:P31/wdt:279* wd:Q83370 . }}
                MINUS {{ ?country wdt:P31/wdt:279* wd:Q7278 . }}

                # Temporal overlap condition with the block
                BIND(COALESCE(?endDate, "{self.end_year}-12-31T23:59:59Z"^^xsd:dateTime) as ?effEndDate)
                FILTER(?startDate < "{chunk_end}-01-01T00:00:00Z"^^xsd:dateTime && ?effEndDate >= "{chunk_start}-01-01T00:00:00Z"^^xsd:dateTime)

                SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
            }}
            LIMIT 5000
            """

            try:
                res = requests.get(url, params={'query': sparql_query, 'format': 'json'}, headers=headers, timeout=30)
                if res.status_code == 200:
                    data = res.json()
                    for item in data['results']['bindings']:
                        country = item['countryLabel']['value']
                        conflict = item['conflictLabel']['value']

                        in_y = self._parse_year(item['startDate']['value'])
                        raw_end = item['endDate']['value'] if 'endDate' in item else None
                        fi_y = self._parse_year(raw_end) if raw_end else None

                        if in_y is None:
                            continue

                        if fi_y is None:
                            fi_y = self.end_year if in_y >= 1800 else (in_y + 5)
                        else:
                            fi_y = max(fi_y, in_y)

                        c_start_y = self._parse_year(item['countryStart']['value']) if 'countryStart' in item else None
                        c_end_y = self._parse_year(item['countryEnd']['value']) if 'countryEnd' in item else (
                            self._parse_year(item['countryEndFallback']['value']) if 'countryEndFallback' in item else None
                        )

                        all_raw.append({
                            'country': country,
                            'conflict_name': conflict,
                            'start_year': in_y,
                            'end_year': fi_y,
                            'country_start': c_start_y,
                            'country_end': c_end_y
                        })
            except Exception as e:
                print(f"⚠️ Block {chunk_start}-{chunk_end} temporarily skipped: {e}")
                continue

        if not all_raw:
            print("❌ No data extracted.")
            return

        df_raw = pd.DataFrame(all_raw).drop_duplicates()
        df_raw = df_raw[~df_raw['country'].str.contains(r'^Q\d+$', na=False)]

        # LIST OF HISTORICAL ENTITIES WITH REAL END YEARS (Safety enforcement for Wikidata gaps)
        forced_max_years = {
            "German Empire": 1918,
            "Qing dynasty": 1912,
            "Kingdom of Montenegro": 1918,
            "Principality of Montenegro": 1910,
            "Russian Soviet Federative Socialist Republic": 1991
        }

        def clean_historical_anachronisms(row):
            c_name = row['country']
            in_y = row['start_year']
            fi_y = row['end_year']

            # If the state is in the list of historical deceased states, the war cannot happen after its end
            if c_name in forced_max_years:
                dead_year = forced_max_years[c_name]
                if in_y > dead_year or fi_y > dead_year:
                    return False

            # Standard sanity filters
            if fi_y < self.start_year or in_y > self.end_year:
                return False

            return True

        df_raw = df_raw[df_raw.apply(clean_historical_anachronisms, axis=1)]

        # RIGOROUS CHRONOLOGICAL AND HISTORICAL SANITY FILTER
        def is_valid_historical_record(row):
            in_y = row['start_year']
            fi_y = row['end_year']
            c_start = row['country_start']
            c_end = row['country_end']

            # 1. Discard wars ending before the dataset start
            if fi_y < self.start_year:
                return False

            # 2. Discard conflicts with future start dates
            if in_y > self.end_year:
                return False

            # 3. State lifespan consistency check (if dates are present in Wikidata)
            if c_start and fi_y < c_start:
                return False
            if c_end and in_y > c_end:
                return False

            return True

        df_raw = df_raw[df_raw.apply(is_valid_historical_record, axis=1)]

        extracted_countries = sorted(list(df_raw['country'].unique()))
        print(f"📊 Extracted {len(df_raw)} valid records (post-anachronism filtering) for {len(extracted_countries)} States. Creating grid...")

        rows = []
        for c in extracted_countries:
            for y in range(self.start_year, self.end_year + 1):
                rows.append({
                    "Country": c,
                    "Year": y,
                    "War_State": 0,
                    "Active_Conflicts": ""
                })

        self.df_long = pd.DataFrame(rows)

        for _, row in df_raw.iterrows():
            c = row["country"]
            in_y = max(row["start_year"], self.start_year)
            fi_y = min(row["end_year"], self.end_year)
            name = row["conflict_name"]

            mask = (
                (self.df_long["Country"] == c) &
                (self.df_long["Year"] >= in_y) &
                (self.df_long["Year"] <= fi_y)
            )

            self.df_long.loc[mask, "War_State"] = 1
            self.df_long.loc[mask, "Active_Conflicts"] = self.df_long.loc[mask, "Active_Conflicts"].apply(
                lambda x: f"{x}; {name}".strip("; ") if x else name
            )

        print("✅ Dynamic dataset refined, cleaned, and successfully loaded!")

    def __call__(self, countries=None, view="long", from_date=None, to_date=None, only_in_war=False, keyword=None):
        """Interface gwd(...) with LIKE support, temporal filters, and advanced views."""
        if self.df_long is None:
            raise ValueError("Dataset not initialized.")
        df = self.df_long.copy()

        if countries:
            c_list = [countries] if isinstance(countries, str) else countries
            regex_pattern = "|".join([re.escape(c) for c in c_list])
            df = df[df['Country'].str.contains(regex_pattern, case=False, na=False)]

        if from_date:
            df = df[df['Year'] >= int(str(from_date).split('-')[0])]
        if to_date:
            df = df[df['Year'] <= int(str(to_date).split('-')[0])]
        if only_in_war:
            df = df[df['War_State'] == 1]
        if keyword:
            df = df[df['Active_Conflicts'].str.contains(keyword, case=False, na=False)]

        if view == "long":
            return df
        elif view == "binary":
            return df.pivot(index="Country", columns="Year", values="War_State").fillna(0).astype(int)
        elif view == "names":
            return df.pivot(index="Country", columns="Year", values="Active_Conflicts").fillna("")
        elif view == "list":
            return sorted(list(set(item for s in df['Active_Conflicts'].dropna() for item in s.split("; ") if item)))
        elif view == "countries":
            active_df = df[df['War_State'] == 1]
            return sorted(list(active_df['Country'].unique()))
        elif view == "conflicts":
            active_df = df[df['War_State'] == 1]
            unique_conflicts = set()
            for s in active_df['Active_Conflicts'].dropna():
                if s:
                    for item in s.split("; "):
                        if item:
                            unique_conflicts.add(item)
            return sorted(list(unique_conflicts))
        else:
            raise ValueError("Invalid view. Choose from: 'long', 'binary', 'names', 'list', 'countries', 'conflicts'")

    @property
    def available_countries(self):
        return sorted(list(self.df_long['Country'].unique())) if self.df_long is not None else []

    def plot_heatmap(self, top_n=20, from_year=1700, to_year=2026):
        df_range = self.df_long[(self.df_long['Year'] >= from_year) & (self.df_long['Year'] <= to_year)]
        top_countries = df_range.groupby("Country")["War_State"].sum().nlargest(top_n).index.tolist()

        df_filtered = df_range[df_range["Country"].isin(top_countries)]

        matrix_val = df_filtered.pivot(index="Country", columns="Year", values="War_State").fillna(0)
        matrix_txt = df_filtered.pivot(index="Country", columns="Year", values="Active_Conflicts").fillna("No recorded conflicts")

        fig = go.Figure(data=go.Heatmap(
            z=matrix_val.values,
            x=matrix_val.columns,
            y=matrix_val.index,
            text=matrix_txt.values,
            hovertemplate="<b>Country:</b> %{y}<br><b>Year:</b> %{x}<br><b>Conflict(s):</b> %{text}<extra></extra>",
            colorscale=[[0, "#ffffff"], [1, "#d9534f"]],
            showscale=False
        ))

        fig.update_layout(
            title=f"Global War State from Wikidata (Sovereign + Historical States) ({from_year}-{to_year})",
            xaxis_title="Year",
            yaxis_title="Country / Historical State",
            xaxis=dict(dtick=10),
            height=700
        )
        fig.show()

    def export_csv(self, query_result, filename="gwd_export.csv"):
        query_result.to_csv(filename)
        files.download(filename)

# Initialize instance with query extended to historical states
gwd = HistWarDB(start_year=1700, end_year=2026)

In [ ]:
gwd.plot_heatmap(top_n=15)

In [97]:
countries_at_war = gwd(from_date=1939, to_date=1945, view="countries")
print("1939-1945", countries_at_war)

conflicts_18th_century = gwd(from_date=1700, to_date=1799, view="conflicts")
print("1700-1799", conflicts_18th_century)

conflicts_18th_century = gwd(from_date=2000, to_date=2026, view="conflicts")
print("2000-2026", conflicts_18th_century)

conflicts_18th_century = gwd(from_date=2000, to_date=2026, view="countries")
print("2000-2026", conflicts_18th_century)

1939-1945 ['Albanian Kingdom', 'Australia', 'Bulgaria', 'Denmark', 'Ecuador', 'Empire of Japan', 'Fascist Italy', 'Finland', 'France', 'Francoist Spain', 'Hungary', 'Indonesia', 'Italian protectorate of Albania', 'Japan', 'Kingdom of Bulgaria', 'Kingdom of Greece', 'Kingdom of Hungary', 'Kingdom of Iraq', 'Kingdom of Italy', 'Kingdom of Romania', 'Kingdom of the Netherlands', 'Manchukuo', "Mongolian People's Republic", 'Nazi Germany', 'Peru', 'Republic of China', 'Russian Soviet Federative Socialist Republic', 'Slovak Republic', 'Soviet Union', 'Taiwan', 'Thailand', 'United Kingdom', 'Vichy France']
1700-1799 ['American-Algerian War', 'Anglo-French War of 1778–1783', 'Anglo-Spanish War', 'Anglo-Spanish War of 1727–1729', 'Apostate War', 'Australian frontier wars', 'Austro-Turkish War of 1716–1718', 'Austro-Turkish War of 1788–1791', 'Burmese–Siamese War of 1775–76', 'Burmese–Siamese War of 1785–86', 'Carnatic Wars', 'Caucasian War', 'Chinese expedition to Tibet', 'Corsican War', 'Dutch

In [69]:
gwd(countries=["United States", "Italy", "China", "France"], from_date=2000, to_date=2026, view="names")

Year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
Country,,,,,,,,,,,,,,,,,,,,,
France,,War in Afghanistan (2001–2021); War on Terrorism,War in Afghanistan (2001–2021); War on Terrori...,War in Afghanistan (2001–2021); War on Terrori...,War in Afghanistan (2001–2021); War on Terrori...,War in Afghanistan (2001–2021); War on Terrori...,War in Afghanistan (2001–2021); War on Terrori...,War in Afghanistan (2001–2021); War on Terrori...,War in Afghanistan (2001–2021); War on Terrori...,War in Afghanistan (2001–2021); War on Terrori...,...,War in Afghanistan (2001–2021); War on Terrori...,War in Afghanistan (2001–2021); War on Terrori...,War in Afghanistan (2001–2021); War on Terrori...,War in Afghanistan (2001–2021); War on Terrori...,War in Afghanistan (2001–2021); War on Terrori...,War on Terrorism; Insurgency in the Maghreb,War on Terrorism; Insurgency in the Maghreb,War on Terrorism; Insurgency in the Maghreb,War on Terrorism; Insurgency in the Maghreb,War on Terrorism; Insurgency in the Maghreb
Italy,,War in Afghanistan (2001–2021),War in Afghanistan (2001–2021),War in Afghanistan (2001–2021); Iraq War,War in Afghanistan (2001–2021); Iraq War,War in Afghanistan (2001–2021); Iraq War,War in Afghanistan (2001–2021); Iraq War,War in Afghanistan (2001–2021); Iraq War,War in Afghanistan (2001–2021); Iraq War,War in Afghanistan (2001–2021); Iraq War,...,War in Afghanistan (2001–2021); War in Afghani...,War in Afghanistan (2001–2021); War in Afghani...,War in Afghanistan (2001–2021); War in Afghani...,War in Afghanistan (2001–2021); War in Afghani...,War in Afghanistan (2001–2021); War in Afghani...,,,,,
United States,,War on Terrorism; War in Afghanistan (2001–2021),War on Terrorism; War in Afghanistan (2001–2021),Iraq War; War on Terrorism; War in Afghanistan...,insurgency in Khyber Pakhtunkhwa; Iraq War; Wa...,insurgency in Khyber Pakhtunkhwa; Iraq War; Wa...,insurgency in Khyber Pakhtunkhwa; Iraq War; Wa...,Iraq War troop surge of 2007; insurgency in Kh...,Iraq War troop surge of 2007; insurgency in Kh...,insurgency in Khyber Pakhtunkhwa; Iraq War; Wa...,...,insurgency in Khyber Pakhtunkhwa; War in Afgha...,insurgency in Khyber Pakhtunkhwa; War in Afgha...,insurgency in Khyber Pakhtunkhwa; War in Afgha...,insurgency in Khyber Pakhtunkhwa; War in Afgha...,insurgency in Khyber Pakhtunkhwa; War in Afgha...,insurgency in Khyber Pakhtunkhwa; War on Terro...,insurgency in Khyber Pakhtunkhwa; War on Terro...,insurgency in Khyber Pakhtunkhwa; War on Terro...,insurgency in Khyber Pakhtunkhwa; War on Terro...,insurgency in Khyber Pakhtunkhwa; War on Terro...
